# 🚀 Laya Decision Engine - Kaggle Server with Ngrok (All 3 Checkpoints Preloaded)

Notebook để chạy **Laya Server** trên Kaggle (GPU T4/P100 hoặc CPU) và public qua **ngrok** với static domain cố định.
Cài đặt trực tiếp từ fork repository [binhminhanh1235/laya-s1-decision](https://github.com/binhminhanh1235/laya-s1-decision).

### ⚠️ Yêu cầu trước khi chạy:
1. **Bật Internet**: Tại thanh bên phải Notebook -> chọn **Internet On**.
2. **Thêm Secret**: Menu trên cùng -> Add-ons -> Secrets -> thêm secret với Label `NGROK_AUTHTOKEN` và Value là auth token Ngrok của bạn.


In [ ]:
# =============================================================================
# 1 CELL DUY NHẤT: CÀI ĐẶT, KẾT NỐI NGROK & KHỞI CHẠY LAYA SERVER (3 CHECKPOINTS)
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

# 1. Cài đặt pyngrok và laya kèm serve & mcp
!pip install -q pyngrok "git+https://github.com/binhminhanh1235/laya-s1-decision.git#egg=laya[serve,mcp]"

from pyngrok import ngrok

# 2. Lấy Ngrok Auth Token từ Kaggle Secrets & cấu hình
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# Đóng tunnel cũ (nếu có khi rerun lại cell) để tránh xung đột domain tĩnh
ngrok.kill()

# 3. Khởi tạo ngrok tunnel với domain cố định
public_url = ngrok.connect(8000, domain="ferret-above-formerly.ngrok-free.app")
separator = "=" * 55
print(f"\n{separator}")
print(f"👉 Laya Server Domain: {public_url}")
print(f"👉 Health Check URL:  {public_url}/health")
print(f"👉 Inference API:      {public_url}/v1/systemone")
    print(f"👉 MCP SSE Endpoint:  {public_url}/sse")\nprint("👉 Loaded Models:      english, multilingual, typed-decisions (Preloaded)")
print("👉 Auto-task route:    Enabled (Coding/Dev tasks -> typed-decisions)")
print(f"{separator}\n")

# 4. Cấu hình Nạp cả 3 checkpoints vào bộ nhớ
os.environ["LAYA_PRELOAD"] = "1"
os.environ["LAYA_MODELS"] = "english,multilingual,typed-decisions"
os.environ["LAYA_MAX_LOADED"] = "3"
os.environ["LAYA_AUTO_TASK"] = "1"
os.environ["LAYA_PORT"] = "8000"

# 5. Chạy Laya server trên port 8000
!python -m laya.serve
